# Time Value of Money: From Basics to Bonds

**Level:** Beginner  
**Tags:** TVM, NPV, IRR, Bond Pricing, Discounting  
**Estimated Time:** 90-120 minutes

## Overview

Master the fundamental concept in finance: a dollar today is worth more than a dollar tomorrow. This notebook covers present value, future value, NPV, IRR, annuities, perpetuities, and bond pricing.

## Learning Objectives

✓ Understand the core principle of time value of money  
✓ Calculate present and future values  
✓ Compute Net Present Value (NPV) for investment decisions  
✓ Calculate Internal Rate of Return (IRR)  
✓ Value annuities and perpetuities  
✓ Price bonds and calculate yield to maturity  
✓ Apply TVM concepts to real-world scenarios  

## Prerequisites

- Basic algebra
- Understanding of percentages and compound interest
- Basic Python (helpful but not required)

## References

1. Brealey, R.A., Myers, S.C., & Allen, F. (2020). *Principles of Corporate Finance*. McGraw-Hill. (Chapters 2-3)
2. Bodie, Z., Kane, A., & Marcus, A.J. (2021). *Investments*. McGraw-Hill. (Chapter 14)
3. Hull, J.C. (2022). *Options, Futures, and Other Derivatives*. Pearson. (Chapter 4)
4. Fabozzi, F.J. (2012). *Bond Markets, Analysis, and Strategies*. Pearson.

In [ ]:
# Import libraries\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom scipy import stats\nfrom scipy.optimize import minimize\nimport warnings\nwarnings.filterwarnings('ignore')\n\n# Configuration\nplt.style.use('seaborn-v0_8-darkgrid')\nsns.set_palette('husl')\n%matplotlib inline\nnp.random.seed(42)\n\nprint('Libraries imported successfully!')

## 1. Introduction: Why Time Value of Money Matters

### The Core Principle

**"A dollar today is worth more than a dollar tomorrow"**

Why? Three main reasons:

1. **Opportunity Cost** - Money today can be invested to earn returns
2. **Inflation** - Future dollars have less purchasing power
3. **Risk/Uncertainty** - Future payments are uncertain

### Real-World Applications

**Corporate Finance:**
- Capital budgeting (should we build that factory?)
- M&A valuation (what's this company worth?)
- Lease vs. buy decisions

**Investments:**
- Bond pricing and yield calculations
- Stock valuation (dividend discount models)
- Real estate investment analysis

**Personal Finance:**
- Mortgage decisions
- Retirement planning
- Education savings

### Historical Context

The concept dates back centuries, but modern formalization came from:
- **Irving Fisher** (1907) - "The Theory of Interest"
- **John Burr Williams** (1938) - Dividend discount model
- **Franco Modigliani & Merton Miller** (1958) - Corporate finance foundations

### What We'll Cover

1. **Basic Concepts** - PV, FV, discount rates
2. **Cash Flow Streams** - Annuities, perpetuities
3. **Investment Criteria** - NPV, IRR, payback period
4. **Bond Pricing** - YTM, duration, convexity
5. **Applications** - Real-world examples

Let's start with the foundations!

In [ ]:
# Quick demonstration of TVM concept
# Which would you prefer: $100 today or $100 in 1 year?

amount = 100
rate = 0.05  # 5% interest rate
years = 1

# Option 1: Take $100 today and invest it
option1_value = amount * (1 + rate) ** years

# Option 2: Wait 1 year for $100
option2_value = 100

print("Time Value of Money Demonstration")
print("=" * 60)
print(f"\nScenario: You can receive $100 today OR $100 in 1 year")
print(f"Assumed interest rate: {rate*100}%\n")
print(f"Option 1: Take ${amount} today")
print(f"  → Invest for {years} year(s) at {rate*100}%")
print(f"  → Future value = ${option1_value:.2f}\n")
print(f"Option 2: Wait {years} year(s) for ${option2_value:.2f}")
print(f"\nDifference: ${option1_value - option2_value:.2f}")
print(f"Conclusion: Take the money TODAY! (worth ${option1_value:.2f} vs ${option2_value:.2f})")

# Visualize the concept
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Timeline visualization
times = [0, 1]
values_option1 = [amount, option1_value]
values_option2 = [0, option2_value]

ax1.plot(times, values_option1, 'o-', linewidth=3, markersize=12, label=f'Option 1: ${amount} today', color='green')
ax1.plot(times, values_option2, 'o-', linewidth=3, markersize=12, label=f'Option 2: ${option2_value} in 1 year', color='red')
ax1.set_xlabel('Time (years)', fontsize=12)
ax1.set_ylabel('Value ($)', fontsize=12)
ax1.set_title('Time Value of Money Comparison', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xticks([0, 1])

# Growth visualization
years_range = np.arange(0, 11)
future_values = amount * (1 + rate) ** years_range

ax2.plot(years_range, future_values, linewidth=3, marker='o', color='steelblue')
ax2.fill_between(years_range, amount, future_values, alpha=0.3, color='steelblue')
ax2.axhline(y=amount, color='red', linestyle='--', alpha=0.5, label=f'Initial ${amount}')
ax2.set_xlabel('Years', fontsize=12)
ax2.set_ylabel('Future Value ($)', fontsize=12)
ax2.set_title(f'Growth of ${amount} at {rate*100}% Annual Interest', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAfter 10 years: ${amount} → ${amount * (1 + rate)**10:.2f}")

## 2. Mathematical Foundation

### 2.1 Future Value (FV)

**Definition:** The value of a current amount at a future date, given a certain interest rate.

**Single Period:**
$$FV = PV \times (1 + r)$$

**Multiple Periods (Compound Interest):**
$$FV = PV \times (1 + r)^n$$

Where:
- $FV$ = Future Value
- $PV$ = Present Value (amount today)
- $r$ = interest rate per period
- $n$ = number of periods

**Continuous Compounding:**
$$FV = PV \times e^{r \times t}$$

---

### 2.2 Present Value (PV)

**Definition:** The current value of a future amount, discounted at a certain rate.

$$PV = \frac{FV}{(1 + r)^n}$$

Or equivalently:
$$PV = FV \times (1 + r)^{-n}$$

**Interpretation:** "How much would I need to invest today to have $FV$ in $n$ periods?"

---

### 2.3 Net Present Value (NPV)

**Definition:** The sum of present values of all cash flows (inflows and outflows).

$$NPV = \sum_{t=0}^{T} \frac{CF_t}{(1 + r)^t}$$

Where:
- $CF_t$ = cash flow at time $t$
- $r$ = discount rate (required return)
- $t$ = time period

**Decision Rule:**
- $NPV > 0$ → **Accept** (creates value)
- $NPV < 0$ → **Reject** (destroys value)
- $NPV = 0$ → **Indifferent** (breaks even)

---

### 2.4 Internal Rate of Return (IRR)

**Definition:** The discount rate that makes NPV = 0.

$$0 = \sum_{t=0}^{T} \frac{CF_t}{(1 + IRR)^t}$$

**Decision Rule:**
- $IRR > r$ (required return) → **Accept**
- $IRR < r$ → **Reject**

**Limitations:**
- Multiple IRRs possible for non-conventional cash flows
- Scale problem (doesn't account for project size)
- Reinvestment assumption

---

### 2.5 Annuities

**Definition:** Stream of equal payments at regular intervals.

**Present Value of Ordinary Annuity:**
$$PV_{annuity} = PMT \times \frac{1 - (1 + r)^{-n}}{r}$$

**Future Value of Ordinary Annuity:**
$$FV_{annuity} = PMT \times \frac{(1 + r)^n - 1}{r}$$

**Annuity Due** (payment at beginning of period):
$$PV_{due} = PV_{ordinary} \times (1 + r)$$

Where:
- $PMT$ = payment per period
- $r$ = interest rate per period
- $n$ = number of periods

---

### 2.6 Perpetuities

**Definition:** Infinite stream of equal payments.

$$PV_{perpetuity} = \frac{PMT}{r}$$

**Growing Perpetuity:**
$$PV_{growing} = \frac{PMT}{r - g}$$

Where $g$ = growth rate (must be $< r$)

**Example:** Preferred stock paying constant dividend forever

---

### 2.7 Bond Pricing

**Zero-Coupon Bond:**
$$P = \frac{F}{(1 + r)^n}$$

**Coupon Bond:**
$$P = \sum_{t=1}^{n} \frac{C}{(1 + r)^t} + \frac{F}{(1 + r)^n}$$

Or more compactly:
$$P = C \times \frac{1 - (1 + r)^{-n}}{r} + \frac{F}{(1 + r)^n}$$

Where:
- $P$ = bond price
- $C$ = coupon payment
- $F$ = face value (par value)
- $r$ = yield to maturity (YTM)
- $n$ = number of periods to maturity

**Yield to Maturity (YTM):**
The discount rate that makes bond price equal to present value of cash flows. Solve for $r$ in the equation above (requires numerical methods).

---

### 2.8 Duration and Convexity

**Macaulay Duration:** Weighted average time to receive cash flows
$$D = \frac{\sum_{t=1}^{n} t \times PV(CF_t)}{P}$$

**Modified Duration:** Price sensitivity to yield changes
$$D_{mod} = \frac{D}{1 + r}$$

**Convexity:** Curvature of price-yield relationship
$$C = \frac{1}{P} \sum_{t=1}^{n} \frac{t(t+1) \times CF_t}{(1+r)^{t+2}}$$

**Price Change Approximation:**
$$\Delta P \approx -D_{mod} \times \Delta r \times P + \frac{1}{2} C \times (\Delta r)^2 \times P$$

## 3. Python Implementation

## 4. Visualization and Analysis

## 5. Practical Applications

## Summary\n\nThis notebook covered time value of money drills. Key takeaways:\n\n- Practical implementation with Python\n- Visualizations and interpretations\n- Real-world applications\n- Best practices and pitfalls\n\n### Next Steps\n\n- Practice with real market data\n- Explore related topics\n- Build your own variations\n\n### Additional Resources\n\n- [QuantEdX Courses](https://www.quantedx.com/courses)\n- [Community Forum](https://www.quantedx.com/community)\n- [GitHub Repository](https://github.com/quantedx)